# Exercice 2 : L'inventaire des actifs

> **En une phrase :** on ne protège pas ce qu'on ne connaît pas : l'inventaire officiel doit correspondre à **ce qui tourne vraiment** sur le réseau.

**Les demandes de l'auditeur traitées ici :** D-02 (inventaire des actifs, **critique**). Références : ISO 27001 A.5.9, CIS Controls v8 1.1.

**Le scénario.** Le TI tient un inventaire (une « CMDB ») dans un tableur. L'auditeur ne le croira pas sur parole :
il demande qu'on le **rapproche du réseau réel**. Lundi, l'équipe a lancé un scan `nmap` de tout le réseau interne.
On compare les deux listes.

**Le plan.**
1. Lire l'inventaire officiel.
2. Lire le résultat brut du scan nmap.
3. Rapprocher les deux listes.
4. Donner un statut à chaque actif.
5. À vous : nommer un propriétaire pour les actifs inconnus.
6. Déposer la pièce.

## Étape 0 : exécutez cette cellule en premier
En local, ouvrez **le dossier du chapitre** dans VS Code. Sur Colab, la cellule télécharge le dossier. Elle doit afficher **Prêt : True**.

In [1]:
# Étape 0 : sur Colab, récupère le dossier du chapitre depuis GitHub. En local (VS Code), cette cellule ne fait rien.
import os
if not os.path.exists('auditeur.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/Code-8INF857---2026.git /content/cours
    %cd "/content/cours/Chapitre 8 - Cadres réglementaires"
print('Prêt :', os.path.exists('auditeur.py'))

Prêt : True


## Étape 1 : L'inventaire officiel
**Ce qu'on fait :** on lit la CMDB du TI.

In [2]:
from helper import lire_csv, lire_texte, deposer
import pandas as pd

cmdb = lire_csv('cmdb_inventaire.csv')
cmdb

,ip,hote,type,os,proprietaire,donnees_personnelles,criticite
0,10.0.1.10,web01,serveur,Ubuntu 22.04,Marc Gagnon (TI),oui,haute
1,10.0.1.11,web02,serveur,Ubuntu 22.04,Marc Gagnon (TI),oui,haute
2,10.0.1.20,bd01,serveur,Ubuntu 20.04,Marc Gagnon (TI),oui,haute
3,10.0.1.30,ftp01,serveur,Debian 10,Marc Gagnon (TI),oui,moyenne
4,10.0.1.40,sauvegarde01,serveur,Ubuntu 22.04,Marc Gagnon (TI),oui,haute
5,10.0.2.5,pare-feu,réseau,pfSense 2.7,Marc Gagnon (TI),non,haute
6,10.0.3.21,poste-rh01,poste,Windows 11,Julie Tremblay (RH),oui,moyenne
7,10.0.3.22,poste-compta01,poste,Windows 11,Luc Côté (Finances),oui,moyenne
8,10.0.3.30,poste-dev01,poste,Ubuntu 24.04,Sami Haddad (Dév.),non,basse
9,10.0.3.31,poste-dev02,poste,Ubuntu 24.04,Sami Haddad (Dév.),non,basse


**👀 Ce qu'on doit voir :** 10 actifs, chacun avec un propriétaire, l'indication « données personnelles » et une criticité. Sur papier, tout est en ordre.

## Étape 2 : Le scan du réseau
**Ce qu'on fait :** on regarde le fichier brut produit par `nmap -oG` (format « grepable »). Une ligne `Status` par hôte actif, une ligne `Ports` avec les ports ouverts.

In [3]:
print('\n'.join(lire_texte('scan_2026-11-23.gnmap').splitlines()[:5]))

# Nmap 7.94 scan initiated Mon Nov 23 09:12:41 2026 as: nmap -sS -sV -oG scan_2026-11-23.gnmap 10.0.0.0/16
Host: 10.0.1.10 (web01.nordheb.local)	Status: Up
Host: 10.0.1.10 (web01.nordheb.local)	Ports: 22/open/tcp//ssh///, 80/open/tcp//http///, 443/open/tcp//https///	Ignored State: closed (997)
Host: 10.0.1.11 (web02.nordheb.local)	Status: Up
Host: 10.0.1.11 (web02.nordheb.local)	Ports: 22/open/tcp//ssh///, 80/open/tcp//http///, 443/open/tcp//https///	Ignored State: closed (997)


**Ce qu'on fait :** on transforme ces lignes en tableau avec une expression régulière : l'IP, le nom DNS et la liste des ports ouverts.

In [4]:
import re

hotes = []
for ligne in lire_texte('scan_2026-11-23.gnmap').splitlines():
    m = re.match(r"Host: (\S+) \((.*?)\)\tPorts: (.*?)\t", ligne)
    if m:
        ip, nom, ports = m.groups()
        hotes.append({'ip': ip, 'nom_dns': nom, 'ports': ' '.join(p.split('/')[0] + '/' + p.split('/')[4] for p in ports.split(', '))})
scan = pd.DataFrame(hotes)
scan

,ip,nom_dns,ports
0,10.0.1.10,web01.nordheb.local,22/ssh 80/http 443/https
1,10.0.1.11,web02.nordheb.local,22/ssh 80/http 443/https
2,10.0.1.20,bd01.nordheb.local,22/ssh 5432/postgresql
3,10.0.1.40,sauvegarde01.nordheb.local,22/ssh
4,10.0.1.47,,139/netbios-ssn 445/microsoft-ds 5000/upnp
5,10.0.2.5,pare-feu.nordheb.local,443/https
6,10.0.3.21,poste-rh01.nordheb.local,135/msrpc 445/microsoft-ds
7,10.0.3.22,poste-compta01.nordheb.local,135/msrpc 445/microsoft-ds
8,10.0.3.30,poste-dev01.nordheb.local,22/ssh
9,10.0.3.31,poste-dev02.nordheb.local,22/ssh


**👀 Ce qu'on doit voir :** 11 hôtes actifs. Deux n'ont pas de nom connu : `10.0.1.47` (pas de nom DNS) et `10.0.3.58` (`DESKTOP-7KQ2M1`, un nom Windows par défaut).

## Étape 3 : Rapprocher les deux listes
**Ce qu'on fait :** une **jointure externe** sur l'IP. L'option `indicator=True` de pandas dit, pour chaque IP, si elle est dans la CMDB seulement, dans le scan seulement, ou dans les deux.

In [5]:
rapprochement = cmdb.merge(scan, on='ip', how='outer', indicator=True)
rapprochement['source'] = rapprochement['_merge'].map({'both': 'cmdb+scan', 'left_only': 'cmdb', 'right_only': 'scan'})
rapprochement[['ip', 'hote', 'nom_dns', 'ports', 'source']].sort_values('source')

,ip,hote,nom_dns,ports,source
3,10.0.1.30,ftp01,NaN,NaN,cmdb
5,10.0.1.47,NaN,,139/netbios-ssn 445/microsoft-ds 5000/upnp,scan
11,10.0.3.58,NaN,DESKTOP-7KQ2M1,135/msrpc 3389/ms-wbt-server,scan
0,10.0.1.10,web01,web01.nordheb.local,22/ssh 80/http 443/https,cmdb+scan
1,10.0.1.11,web02,web02.nordheb.local,22/ssh 80/http 443/https,cmdb+scan
2,10.0.1.20,bd01,bd01.nordheb.local,22/ssh 5432/postgresql,cmdb+scan
4,10.0.1.40,sauvegarde01,sauvegarde01.nordheb.local,22/ssh,cmdb+scan
6,10.0.2.5,pare-feu,pare-feu.nordheb.local,443/https,cmdb+scan
7,10.0.3.21,poste-rh01,poste-rh01.nordheb.local,135/msrpc 445/microsoft-ds,cmdb+scan
8,10.0.3.22,poste-compta01,poste-compta01.nordheb.local,135/msrpc 445/microsoft-ds,cmdb+scan


**👀 Ce qu'on doit voir :**
- **2 actifs dans le scan seulement** : ils tournent sur le réseau, mais personne ne les a déclarés ;
- **1 actif dans la CMDB seulement** : `ftp01` (10.0.1.30). Il est dans l'inventaire, mais n'a pas répondu au scan.

**❓ Question pour la classe :** d'après leurs ports, que sont ces deux machines inconnues ?

*Piste : `10.0.1.47` a les ports 139 et 445 (partage de fichiers Windows) et 5000 (l'interface Web des NAS Synology) : c'est sans doute un **NAS** branché par quelqu'un. `10.0.3.58` a le port 3389 ouvert (**bureau à distance**, RDP) et un nom Windows par défaut : un poste personnel ou installé à la va-vite. Deux portes d'entrée que personne ne surveille.*

## Étape 4 : Un statut pour chaque actif
**Ce qu'on fait :** on traduit la source en statut d'audit :
- `cmdb+scan` → **conforme** (déclaré et vu) ;
- `scan` → **non inventorié** (vu, mais jamais déclaré : à identifier, puis inscrire ou retirer) ;
- `cmdb` → **introuvable** (déclaré, mais pas vu : éteint ? débranché ? volé ?).

In [6]:
rapprochement['statut'] = rapprochement['source'].map({'cmdb+scan': 'conforme', 'scan': 'non inventorié', 'cmdb': 'introuvable'})
rapprochement['hote'] = rapprochement['hote'].fillna(rapprochement['nom_dns']).replace('', 'inconnu')
rapprochement[['ip', 'hote', 'source', 'statut', 'proprietaire']].sort_values('statut')

,ip,hote,source,statut,proprietaire
3,10.0.1.30,ftp01,cmdb,introuvable,Marc Gagnon (TI)
5,10.0.1.47,inconnu,scan,non inventorié,NaN
11,10.0.3.58,DESKTOP-7KQ2M1,scan,non inventorié,NaN
0,10.0.1.10,web01,cmdb+scan,conforme,Marc Gagnon (TI)
1,10.0.1.11,web02,cmdb+scan,conforme,Marc Gagnon (TI)
2,10.0.1.20,bd01,cmdb+scan,conforme,Marc Gagnon (TI)
4,10.0.1.40,sauvegarde01,cmdb+scan,conforme,Marc Gagnon (TI)
6,10.0.2.5,pare-feu,cmdb+scan,conforme,Marc Gagnon (TI)
7,10.0.3.21,poste-rh01,cmdb+scan,conforme,Julie Tremblay (RH)
8,10.0.3.22,poste-compta01,cmdb+scan,conforme,Luc Côté (Finances)


**👀 Ce qu'on doit voir :** les deux actifs non inventoriés n'ont **aucun propriétaire** (`NaN`). Pour l'auditeur, un actif sans propriétaire est un actif dont personne n'est responsable.

## À vous : un propriétaire pour chaque actif inconnu
Après enquête, l'équipe a appris :
- le NAS `10.0.1.47` a été acheté par les **RH** pour stocker les CV des candidats ;
- le poste `DESKTOP-7KQ2M1` est l'ordinateur personnel d'un stagiaire du **soutien**, branché au réseau.

Écrivez un propriétaire (une personne et son service) pour ces deux IP. Inspirez-vous de la colonne `proprietaire` de la CMDB.

In [7]:
# À VOUS : un propriétaire (personne et service) pour chaque actif non inventorié
proprietaires_inconnus = {
    '10.0.1.47': '',
    '10.0.3.58': '',
}

*Réponse (une possibilité) :* `'10.0.1.47': 'Julie Tremblay (RH)'` et `'10.0.3.58': 'Marc Gagnon (TI), en attendant le retrait du poste'`.
Le propriétaire n'est pas forcément celui qui a branché la machine : c'est la personne **responsable** de décider de son sort.

## Étape 5 : Déposer la pièce D-02
**Ce qu'on fait :** on complète les propriétaires et on dépose l'inventaire rapproché.

In [8]:
rapprochement['proprietaire'] = rapprochement['proprietaire'].fillna('')
for ip, qui in proprietaires_inconnus.items():
    rapprochement.loc[rapprochement['ip'] == ip, 'proprietaire'] = qui

inventaire = rapprochement[['ip', 'hote', 'type', 'source', 'statut', 'proprietaire', 'ports', 'donnees_personnelles']].fillna('')
deposer('inventaire.csv', inventaire)
inventaire

Pièce déposée : livrables/inventaire.csv


,ip,hote,type,source,statut,proprietaire,ports,donnees_personnelles
0,10.0.1.10,web01,serveur,cmdb+scan,conforme,Marc Gagnon (TI),22/ssh 80/http 443/https,oui
1,10.0.1.11,web02,serveur,cmdb+scan,conforme,Marc Gagnon (TI),22/ssh 80/http 443/https,oui
2,10.0.1.20,bd01,serveur,cmdb+scan,conforme,Marc Gagnon (TI),22/ssh 5432/postgresql,oui
3,10.0.1.30,ftp01,serveur,cmdb,introuvable,Marc Gagnon (TI),,oui
4,10.0.1.40,sauvegarde01,serveur,cmdb+scan,conforme,Marc Gagnon (TI),22/ssh,oui
5,10.0.1.47,inconnu,,scan,non inventorié,,139/netbios-ssn 445/microsoft-ds 5000/upnp,
6,10.0.2.5,pare-feu,réseau,cmdb+scan,conforme,Marc Gagnon (TI),443/https,non
7,10.0.3.21,poste-rh01,poste,cmdb+scan,conforme,Julie Tremblay (RH),135/msrpc 445/microsoft-ds,oui
8,10.0.3.22,poste-compta01,poste,cmdb+scan,conforme,Luc Côté (Finances),135/msrpc 445/microsoft-ds,oui
9,10.0.3.30,poste-dev01,poste,cmdb+scan,conforme,Sami Haddad (Dév.),22/ssh,non


## Ce que dit l'auditeur
**Ce qu'on fait :** on relance l'auditeur automatique sur tout le dossier `livrables/`. D-02 est une demande **critique**.

In [9]:
import auditeur
auditeur.auditer();

Cabinet Boréal Audit — revue des pièces déposées

⚠️ D-01 Textes applicables et justification          9.3 / 10
      · nis2 : applicabilité « ? » incorrecte
      · nis2 : justification manquante ou trop courte
⚠️ D-02 Inventaire des actifs rapproché du réseau   14.2 / 15  (critique)
      · 10.0.1.47 : aucun propriétaire
      · 10.0.3.58 : aucun propriétaire
❌ D-03 Configuration sécurisée des serveurs         0.0 / 15
      · fichier controles.csv introuvable dans livrables/
❌ D-04 Sauvegardes et test de restauration          0.0 / 10  (critique)
      · fichier controles.csv introuvable dans livrables/
❌ D-05 Revue des droits d'accès                     0.0 / 15  (critique)
      · fichier revue_acces.csv introuvable dans livrables/
❌ D-06 Registre et plan de traitement des risques   0.0 / 15
      · fichier registre_risques.csv introuvable dans livrables/
❌ D-07 Registre des incidents et notifications      0.0 / 15  (critique)
      · fichier registre_incidents.csv introuvable dan

**👀 Ce qu'on doit voir :** D-02 à ⚠️ **14,2 / 15** tant que les deux actifs inconnus n'ont pas de propriétaire. Avec votre tâche, ✅ **15 / 15**.
Score : **23 / 100** sans les tâches, **25 / 100** avec les tâches des ex. 1 et 2. La critique D-02 n'est plus refusée.

**❓ Question pour la classe :** `ftp01` est « introuvable ». Que doit-on faire avant la visite de l'auditeur ?

*Piste : savoir **où il est**. Éteint mais toujours branché ? Il contient des données personnelles (CMDB) : s'il a disparu, c'est peut-être un incident de confidentialité (perte). Le registre de risques (ex5) en reparlera : c'est un Debian 10, qui n'a plus de mises à jour de sécurité.*

## Ce qu'il faut retenir
- L'inventaire est la **première** demande de presque tous les cadres (CIS Control 1, ISO A.5.9) : on ne protège pas ce qu'on ne connaît pas.
- Un inventaire déclaré ne vaut rien sans **rapprochement** avec la réalité (scan réseau, console EDR, factures).
- Trois cas : déclaré et vu (conforme), vu mais pas déclaré (**shadow IT**), déclaré mais pas vu (introuvable).
- Chaque actif a un **propriétaire** : la personne qui répond de lui.

**Exercice suivant :** l'auditeur veut maintenant des **preuves** que nos serveurs sont bien configurés.